# LeetCode 500 — Keyboard Row bằng Python

**Link đề:** https://leetcode.com/problems/keyboard-row/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

```python
class Solution:
    def findWords(self, words: list[str]) -> list[str]:
        ...
```

## Mục tiêu học
1. Đọc đề kiểu "lọc danh sách theo điều kiện" và tách thành hàm con `can_type_one_row(word)`.
2. Ôn chuỗi: `str.lower()`, toán tử `in`, duyệt ký tự, `all(...)`.
3. Học 3 công cụ tra cứu: chuỗi (`in` O(k)), `dict` ký tự → hàng, `set` và phép **tập con** `<=`.
4. Đi từ cách trực giác (thử từng hàng) → bảng tra `dict` → phép tập hợp → (tuỳ chọn) regex.
5. Trace, test chữ hoa/thường, so sánh các cách, luyện biến thể.

## 1. Đề bài

Cho mảng chuỗi `words`, trả về các từ **gõ được chỉ bằng chữ cái trên một hàng** của bàn phím Mỹ.

Chữ hoa và chữ thường **coi như nhau** (`'a'` và `'A'` cùng hàng 2).

| Hàng | Ký tự |
|---|---|
| 1 | `qwertyuiop` |
| 2 | `asdfghjkl` |
| 3 | `zxcvbnm` |

| words | Output |
|---|---|
| `["Hello","Alaska","Dad","Peace"]` | `["Alaska","Dad"]` |
| `["omk"]` | `[]` |
| `["adsdf","sfd"]` | `["adsdf","sfd"]` |

**Constraints:** `1 <= words.length <= 20`; `1 <= words[i].length <= 100`; `words[i]` chỉ gồm chữ cái tiếng Anh (hoa và thường).

Lưu ý: kết quả giữ **nguyên dạng chữ gốc** và **thứ tự ban đầu** (`"Alaska"` chứ không phải `"alaska"`).

### Phân tích bằng tay ví dụ 1

| word | lower | hàng của từng ký tự | Cùng một hàng? |
|---|---|---|---|
| Hello | hello | h→2, e→1, l→2, l→2, o→1 | Không |
| Alaska | alaska | a→2, l→2, a→2, s→2, k→2, a→2 | **Có** (hàng 2) |
| Dad | dad | d→2, a→2, d→2 | **Có** (hàng 2) |
| Peace | peace | p→1, e→1, a→2, c→3, e→1 | Không |

Ý chính: **mọi ký tự phải thuộc cùng một hàng** — hàng của ký tự đầu tiên quyết định hàng cần kiểm tra.

## 2. Nền tảng Python

### 2.1. `str.lower()` — đưa về chữ thường

Vì đề không phân biệt hoa/thường, ta chuẩn hoá bằng `lower()`. Hàm trả về **chuỗi mới**; chuỗi gốc không đổi (chuỗi trong Python là **immutable** — bất biến).

In [1]:
w = "Alaska"
print(w.lower())   # alaska
print(w)           # vẫn là Alaska — chuỗi gốc không đổi
print("QwE".lower(), "QwE".upper())

alaska
Alaska
qwe QWE


### 2.2. Toán tử `in` trên chuỗi

`ch in "qwertyuiop"` hỏi "ký tự `ch` có nằm trong chuỗi này không?". Python duyệt chuỗi để tìm → tốn `O(độ dài chuỗi)`, ở đây tối đa 10 nên rất nhanh.

In [2]:
row1 = "qwertyuiop"
print("e" in row1, "a" in row1)
print("ert" in row1)   # với chuỗi, `in` còn kiểm tra CHUỖI CON liên tiếp

True False
True


### 2.3. `all(...)` — "tất cả đều đúng"

`all(iterable)` trả `True` nếu **mọi** phần tử là đúng (và trả `True` với iterable rỗng). Nó **dừng sớm** khi gặp phần tử sai đầu tiên.

Thường dùng với **generator expression**: `all(ch in row for ch in word)`.

In [3]:
row2 = "asdfghjkl"
print(all(ch in row2 for ch in "dad"))     # True
print(all(ch in row2 for ch in "hello"))   # False — dừng ở 'e'
print(all([]))                             # True (không có phản ví dụ nào)
print(any(ch in row2 for ch in "qwe"))     # any: "có ít nhất một"

True
False
True
False


### 2.4. `set` và phép tập con `<=`

`set` là tập hợp **không trùng, không thứ tự**, tra cứu `in` trung bình O(1).
- `set("dad")` → `{'d', 'a'}`
- `A <= B` nghĩa là **A là tập con của B** (mọi phần tử của A đều có trong B). Tương đương `A.issubset(B)`.

In [4]:
row2_set = set("asdfghjkl")
print(set("dad"))
print(set("dad") <= row2_set)          # True
print(set("hello") <= row2_set)        # False
print(set("hello") - row2_set)         # phép hiệu: những ký tự KHÔNG thuộc hàng 2

{'d', 'a'}
True
False
{'e', 'o'}


### 2.5. `dict` ký tự → hàng

Ta có thể dựng một bảng tra `row_of['a'] = 2`... bằng hai vòng lặp, hoặc gọn hơn bằng **dict comprehension**.

In [5]:
rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
row_of = {}
for idx, row in enumerate(rows, start=1):
    for ch in row:
        row_of[ch] = idx
print(len(row_of), "ký tự")
print(row_of["q"], row_of["a"], row_of["m"])

# Viết gọn tương đương:
row_of2 = {ch: idx for idx, row in enumerate(rows, start=1) for ch in row}
print(row_of == row_of2)

26 ký tự
1 2 3
True


## 3. Cách 1 — Trực giác: thử từng hàng

Với mỗi từ, thử lần lượt 3 hàng: nếu mọi ký tự (đã `lower`) đều nằm trong hàng đó → nhận từ.

In [6]:
ROWS = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]

def find_words_try_rows(words):
    result = []
    for word in words:
        w = word.lower()
        for row in ROWS:
            ok = True
            for ch in w:
                if ch not in row:
                    ok = False
                    break
            if ok:
                result.append(word)   # thêm từ GỐC
                break                 # đã khớp 1 hàng, không thử hàng khác
    return result

print(find_words_try_rows(["Hello", "Alaska", "Dad", "Peace"]))  # ['Alaska', 'Dad']
print(find_words_try_rows(["omk"]))                               # []
print(find_words_try_rows(["adsdf", "sfd"]))                      # ['adsdf', 'sfd']

['Alaska', 'Dad']
[]
['adsdf', 'sfd']


### Nút thắt nhỏ của cách 1

- Mỗi từ có thể bị duyệt **3 lần** (một lần cho mỗi hàng).
- Mỗi lần `ch not in row` lại quét chuỗi hàng (dài ≤ 10).

Với constraints nhỏ thì không sao, nhưng ta có thể **chỉ duyệt mỗi từ một lần**: nhìn ký tự đầu để biết **hàng mục tiêu**, rồi kiểm tra các ký tự còn lại có cùng hàng không. Muốn biết "ký tự thuộc hàng nào" nhanh → bảng tra `dict`.

## 4. Insight — hàng của ký tự đầu quyết định tất cả

Nếu từ gõ được trên một hàng, hàng đó **bắt buộc** là hàng của ký tự đầu tiên. Vậy:

```text
target = row_of[w[0]]
từ hợp lệ  <=>  mọi ch trong w có row_of[ch] == target
```

In [7]:
ROW_OF = {ch: idx for idx, row in enumerate(ROWS, start=1) for ch in row}

w = "alaska"
target = ROW_OF[w[0]]
print("hàng mục tiêu:", target)
print([ (ch, ROW_OF[ch]) for ch in w ])

hàng mục tiêu: 2
[('a', 2), ('l', 2), ('a', 2), ('s', 2), ('k', 2), ('a', 2)]


## 5. Cách 2 — Lời giải chính với bảng tra `dict`

Các đoạn:
1. `ROW_OF` dựng **một lần** (ngoài vòng lặp).
2. Với mỗi `word`: `w = word.lower()`, `target = ROW_OF[w[0]]`.
3. `all(ROW_OF[ch] == target for ch in w)` → nếu đúng thì `append(word)`.

In [8]:
def find_words_row_map(words):
    result = []
    for word in words:
        w = word.lower()
        target = ROW_OF[w[0]]
        if all(ROW_OF[ch] == target for ch in w):
            result.append(word)
    return result

print(find_words_row_map(["Hello", "Alaska", "Dad", "Peace"]))
print(find_words_row_map(["omk"]))

['Alaska', 'Dad']
[]


## 6. TRACE từng bước

Ví dụ `["Hello", "Alaska", "Dad", "Peace", "QWERTY", "zXc"]`:

| word | target | ký tự gây dừng | Kết quả |
|---|---:|---|---|
| Hello | 2 (h) | `e` (hàng 1) | loại |
| Alaska | 2 (a) | — | nhận |
| Dad | 2 (d) | — | nhận |
| Peace | 1 (p) | `a` (hàng 2) | loại |
| QWERTY | 1 (q) | — | nhận |
| zXc | 3 (z) | — | nhận |

In [9]:
def trace_find_words(words):
    result = []
    for word in words:
        w = word.lower()
        target = ROW_OF[w[0]]
        print(f"'{word}': lower='{w}', target=hàng {target}")
        ok = True
        for i, ch in enumerate(w):
            r = ROW_OF[ch]
            mark = "ok" if r == target else "KHÁC HÀNG -> dừng"
            print(f"    i={i} ch='{ch}' hàng={r} {mark}")
            if r != target:
                ok = False
                break
        if ok:
            result.append(word)
        print(f"  => {'NHẬN' if ok else 'LOẠI'} | result={result}")
    return result

trace_find_words(["Hello", "Alaska", "Dad", "Peace", "QWERTY", "zXc"])

'Hello': lower='hello', target=hàng 2
    i=0 ch='h' hàng=2 ok
    i=1 ch='e' hàng=1 KHÁC HÀNG -> dừng
  => LOẠI | result=[]
'Alaska': lower='alaska', target=hàng 2
    i=0 ch='a' hàng=2 ok
    i=1 ch='l' hàng=2 ok
    i=2 ch='a' hàng=2 ok
    i=3 ch='s' hàng=2 ok
    i=4 ch='k' hàng=2 ok
    i=5 ch='a' hàng=2 ok
  => NHẬN | result=['Alaska']
'Dad': lower='dad', target=hàng 2
    i=0 ch='d' hàng=2 ok
    i=1 ch='a' hàng=2 ok
    i=2 ch='d' hàng=2 ok
  => NHẬN | result=['Alaska', 'Dad']
'Peace': lower='peace', target=hàng 1
    i=0 ch='p' hàng=1 ok
    i=1 ch='e' hàng=1 ok
    i=2 ch='a' hàng=2 KHÁC HÀNG -> dừng
  => LOẠI | result=['Alaska', 'Dad']
'QWERTY': lower='qwerty', target=hàng 1
    i=0 ch='q' hàng=1 ok
    i=1 ch='w' hàng=1 ok
    i=2 ch='e' hàng=1 ok
    i=3 ch='r' hàng=1 ok
    i=4 ch='t' hàng=1 ok
    i=5 ch='y' hàng=1 ok
  => NHẬN | result=['Alaska', 'Dad', 'QWERTY']
'zXc': lower='zxc', target=hàng 3
    i=0 ch='z' hàng=3 ok
    i=1 ch='x' hàng=3 ok
    i=2 ch='c' hàng=3 o

['Alaska', 'Dad', 'QWERTY', 'zXc']

## 7. Cách 3 — Phép tập hợp (set)

Một từ hợp lệ ⇔ **tập ký tự của nó là tập con của một hàng nào đó**:

```python
s = set(word.lower())
s <= ROW1 or s <= ROW2 or s <= ROW3
```

Gọn, rất "Pythonic". Dùng thêm `any(...)` để viết cho 3 hàng.

In [10]:
ROW_SETS = [set(r) for r in ROWS]

def find_words_set(words):
    return [w for w in words if any(set(w.lower()) <= row for row in ROW_SETS)]

print(find_words_set(["Hello", "Alaska", "Dad", "Peace"]))
print(find_words_set(["adsdf", "sfd"]))

['Alaska', 'Dad']
['adsdf', 'sfd']


`[w for w in words if điều_kiện]` là **list comprehension có lọc**: giữ lại `w` khi điều kiện đúng, đúng thứ tự gốc.

## 8. Cách 4 — Biểu thức chính quy (regex) — để mở rộng tầm nhìn

Module chuẩn `re`: `re.fullmatch(pattern, s)` kiểm tra **toàn bộ** chuỗi khớp pattern.
- `[qwertyuiop]*` = "0 hoặc nhiều ký tự trong nhóm này".
- `|` = "hoặc"; cờ `re.IGNORECASE` bỏ qua hoa/thường.

In [11]:
import re
PATTERN = re.compile(r"[qwertyuiop]*|[asdfghjkl]*|[zxcvbnm]*", re.IGNORECASE)

def find_words_regex(words):
    return [w for w in words if PATTERN.fullmatch(w)]

print(find_words_regex(["Hello", "Alaska", "Dad", "Peace"]))
print(bool(PATTERN.fullmatch("Typewriter")), bool(PATTERN.fullmatch("Hello")))

['Alaska', 'Dad']
True False


## 9. Code nộp LeetCode

### Bản bảng tra `dict` (khuyên dùng — rõ ràng, dừng sớm)

In [12]:
class Solution:
    def findWords(self, words: list[str]) -> list[str]:
        rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
        row_of = {ch: i for i, row in enumerate(rows) for ch in row}
        result = []
        for word in words:
            w = word.lower()
            target = row_of[w[0]]
            if all(row_of[ch] == target for ch in w):
                result.append(word)
        return result

print(Solution().findWords(["Hello", "Alaska", "Dad", "Peace"]))

['Alaska', 'Dad']


### Bản `set` (ngắn gọn)

In [13]:
class SolutionSet:
    def findWords(self, words: list[str]) -> list[str]:
        rows = [set("qwertyuiop"), set("asdfghjkl"), set("zxcvbnm")]
        return [w for w in words if any(set(w.lower()) <= r for r in rows)]

print(SolutionSet().findWords(["Hello", "Alaska", "Dad", "Peace"]))

['Alaska', 'Dad']


(Trên LeetCode, đổi tên `SolutionSet` thành `Solution` khi nộp.)

## 10. Vì sao đúng?

- **Cần:** nếu từ gõ được trên một hàng `R` thì ký tự đầu thuộc `R`, nên `R = row_of[w[0]]` (mỗi chữ cái chỉ thuộc đúng 1 hàng — 3 hàng rời nhau và phủ đủ 26 chữ).
- **Đủ:** nếu mọi ký tự có `row_of[ch] == target` thì cả từ nằm trên hàng `target`.
- Việc `lower()` đảm bảo `'A'` và `'a'` được tra như nhau; ta `append(word)` (bản gốc) nên giữ đúng dạng và thứ tự.
- Cách set: `set(w) <= R` đúng nghĩa đen "mọi ký tự của w thuộc R".

Kiểm chứng giả thiết "3 hàng rời nhau và phủ đủ 26 chữ":

In [14]:
import string
all_chars = "".join(ROWS)
print(len(all_chars), len(set(all_chars)))                 # 26 26 -> không trùng
print(set(all_chars) == set(string.ascii_lowercase))       # True -> đủ 26 chữ

26 26
True


## 11. Độ phức tạp

Đặt `N` = tổng số ký tự của mọi từ (`N ≤ 20·100 = 2000`).

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Thử từng hàng | `O(3 · N · 10)` = `O(N)` | `O(1)` | mỗi `in` quét ≤ 10 ký tự |
| Bảng tra `dict` | `O(N)` | `O(1)` (bảng 26 key) | mỗi ký tự tra 1 lần, dừng sớm |
| Set tập con | `O(N)` | `O(L)` cho `set(w)` | tạo set mỗi từ, không dừng sớm khi tạo set |
| Regex | `O(N)` | `O(1)` | engine regex lo phần lặp |

Mọi cách đều tuyến tính theo tổng độ dài; khác biệt ở hằng số và độ dễ đọc. Kết quả trả về chiếm thêm `O(số từ)` (không tính vào bộ nhớ phụ).

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.findWords(["Hello", "Alaska", "Dad", "Peace"]), ["Alaska", "Dad"])
check("đề 2", sol.findWords(["omk"]), [])
check("đề 3", sol.findWords(["adsdf", "sfd"]), ["adsdf", "sfd"])
check("1 ký tự", sol.findWords(["a", "Q", "m"]), ["a", "Q", "m"])
check("chữ hoa toàn bộ", sol.findWords(["ZXCV", "TYPE", "HELLO"]), ["ZXCV", "TYPE"])
check("hoa thường lẫn lộn", sol.findWords(["qWeRtY", "AsDfG", "zXcVbN"]), ["qWeRtY", "AsDfG", "zXcVbN"])
check("giữ thứ tự & trùng", sol.findWords(["dad", "Hello", "dad"]), ["dad", "dad"])
check("ký tự cuối khác hàng", sol.findWords(["asdfghjklz"]), [])

PASS đề 1: ['Alaska', 'Dad']
PASS đề 2: []
PASS đề 3: ['adsdf', 'sfd']
PASS 1 ký tự: ['a', 'Q', 'm']
PASS chữ hoa toàn bộ: ['ZXCV', 'TYPE']
PASS hoa thường lẫn lộn: ['qWeRtY', 'AsDfG', 'zXcVbN']
PASS giữ thứ tự & trùng: ['dad', 'dad']
PASS ký tự cuối khác hàng: []


In [16]:
import random
random.seed(500)
letters = string.ascii_letters
for _ in range(500):
    words = []
    for _ in range(random.randint(1, 20)):
        if random.random() < 0.5:   # tạo từ cố ý nằm trên 1 hàng
            row = random.choice(ROWS)
            w = "".join(random.choice(row) for _ in range(random.randint(1, 8)))
            w = "".join(c.upper() if random.random() < 0.3 else c for c in w)
        else:
            w = "".join(random.choice(letters) for _ in range(random.randint(1, 8)))
        words.append(w)
    r1 = find_words_try_rows(words)
    r2 = find_words_row_map(words)
    r3 = find_words_set(words)
    r4 = find_words_regex(words)
    r5 = Solution().findWords(words)
    assert r1 == r2 == r3 == r4 == r5, (words, r1, r2, r3, r4, r5)
print("500 bộ ngẫu nhiên: 5 cài đặt khớp nhau — PASS")

500 bộ ngẫu nhiên: 5 cài đặt khớp nhau — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: quên `lower()`
`'H'` không có trong `"asdfghjkl"` → `"Dad"` bị loại sai (vì `'D'` hoa).

### Lỗi 2: trả về bản đã `lower()`
`result.append(w)` thay vì `result.append(word)` → `"alaska"` thay vì `"Alaska"`.

### Lỗi 3: dùng `any` thay vì `all`
"Có ít nhất một ký tự thuộc hàng" — gần như từ nào cũng qua.

### Lỗi 4: quên `break` sau khi khớp một hàng (cách 1)
Không xảy ra với 3 hàng rời nhau, nhưng nếu bảng hàng có chữ trùng (biến thể) thì một từ có thể bị thêm **2 lần**.

In [17]:
def buggy_no_lower(words):
    return [w for w in words if any(set(w) <= set(r) for r in ROWS)]

def buggy_return_lower(words):
    return [w.lower() for w in words if any(set(w.lower()) <= set(r) for r in ROWS)]

def buggy_any(words):
    res = []
    for word in words:
        w = word.lower()
        t = ROW_OF[w[0]]
        if any(ROW_OF[ch] == t for ch in w):
            res.append(word)
    return res

ex = ["Hello", "Alaska", "Dad", "Peace"]
print("quên lower   :", buggy_no_lower(ex))
print("trả bản lower:", buggy_return_lower(ex))
print("any thay all :", buggy_any(ex))
print("đúng         :", Solution().findWords(ex))

quên lower   : []
trả bản lower: ['alaska', 'dad']
any thay all : ['Hello', 'Alaska', 'Dad', 'Peace']
đúng         : ['Alaska', 'Dad']


## 14. Mẹo nhận dạng pattern

- "Mọi phần tử thuộc cùng một nhóm" → **gán nhãn nhóm bằng `dict`** rồi so sánh nhãn với nhãn của phần tử đầu, hoặc **tập con** `set(x) <= group`.
- "Không phân biệt hoa/thường" → chuẩn hoá (`lower()` / `casefold()`) **trước khi** tra cứu, nhưng **trả về bản gốc**.
- "Lọc danh sách giữ thứ tự" → list comprehension `[x for x in arr if ok(x)]`.
- Bảng tra cố định (26 chữ cái, 10 chữ số) → dựng **một lần** ngoài vòng lặp; bộ nhớ được coi là `O(1)`.

## 15. Biến thể

### 15.1. Trả về hàng của mỗi từ (hoặc -1)

In [18]:
def row_of_word(word):
    w = word.lower()
    t = ROW_OF[w[0]]
    return t if all(ROW_OF[ch] == t for ch in w) else -1

print([(w, row_of_word(w)) for w in ["Hello", "Alaska", "Type", "zzz"]])

[('Hello', -1), ('Alaska', 2), ('Type', 1), ('zzz', 3)]


### 15.2. Bàn phím tuỳ ý
Bàn phím là list các chuỗi bất kỳ (có thể không đủ 26 chữ). Từ có ký tự không thuộc hàng nào → loại. Lúc này `ROW_OF[ch]` có thể gây `KeyError`, nên dùng `.get(ch)`.

In [19]:
def find_words_custom(words, rows):
    row_of = {ch: i for i, row in enumerate(rows) for ch in row.lower()}
    res = []
    for word in words:
        w = word.lower()
        t = row_of.get(w[0])
        if t is not None and all(row_of.get(ch) == t for ch in w):
            res.append(word)
    return res

print(find_words_custom(["abc", "Cab", "dd", "xyz"], ["abc", "de"]))  # ['abc', 'Cab', 'dd']

['abc', 'Cab', 'dd']


## 16. Bài tập (đáp án ở cuối)

### Bài 1 — Đoán output (không chạy)
`set("Peace".lower()) <= set("qwertyuiop")` trả gì? Tập `set("Peace".lower()) - set("qwertyuiop")` là gì?

### Bài 2 — Điền TODO (cách dict)

In [20]:
def exercise_find_words(words):
    rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
    row_of = {}
    # TODO 1: điền row_of[ch] = chỉ số hàng
    result = []
    for word in words:
        # TODO 2: lower, lấy target từ ký tự đầu
        # TODO 3: nếu mọi ký tự cùng target -> append từ GỐC
        pass
    return result

# assert exercise_find_words(["Hello", "Alaska", "Dad", "Peace"]) == ["Alaska", "Dad"]

### Bài 3 — Đếm số từ hợp lệ trên mỗi hàng
Trả về list 3 số `[c1, c2, c3]` là số từ gõ được hoàn toàn trên hàng 1, 2, 3.

In [21]:
def count_per_row(words):
    # TODO
    pass

# assert count_per_row(["Hello", "Alaska", "Dad", "Peace", "type", "zxc"]) == [1, 2, 1]

### Bài 4 — Từ dài nhất gõ được trên một hàng
Trả về từ dài nhất (nếu hoà, lấy từ xuất hiện trước); không có thì trả `""`.

In [22]:
def longest_one_row_word(words):
    # TODO
    pass

# assert longest_one_row_word(["Hello", "Alaska", "Dad", "typewriter"]) == "typewriter"
# assert longest_one_row_word(["Hello"]) == ""

### Bài 5 — Đổi hàng ít nhất
Với một từ, đếm số lần **chuyển hàng** khi gõ lần lượt từng ký tự (ví dụ `"hello"`: h(2)→e(1) là 1 lần, e(1)→l(2) là 2, l→l không đổi, l(2)→o(1) là 3).

In [23]:
def row_switches(word):
    # TODO
    pass

# assert row_switches("hello") == 3
# assert row_switches("Dad") == 0

### Bài 6 — Mini challenge
Cho `words`, nhóm các từ gõ được trên một hàng thành dict `{số_hàng: [các từ]}` (chỉ ghi hàng có ít nhất 1 từ).

In [24]:
def group_by_row(words):
    # TODO
    pass

# assert group_by_row(["Alaska", "Dad", "type", "Hello"]) == {2: ["Alaska", "Dad"], 1: ["type"]}

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `a`, `c` có trong hàng 1 không? | phép hiệu giữ phần tử chỉ có ở vế trái | tính `set("peace")` trước rồi so từng phần tử với hàng 1 |
| 2 | Xem mục 5 | `for i, row in enumerate(rows)` | `if all(row_of[ch] == target for ch in w): result.append(word)` |
| 3 | Hàm `row_of_word` ở biến thể 15.1 | list đếm `[0, 0, 0]` | `counts[r - 1] += 1` khi `r != -1` |
| 4 | Lọc trước, so độ dài sau | giữ `best = ""` | chỉ cập nhật khi `len(w) > len(best)` (dấu `>` để giữ từ trước khi hoà) |
| 5 | So sánh hàng của 2 ký tự liền kề | `for i in range(1, len(w))` | `count += row_of[w[i]] != row_of[w[i-1]]` |
| 6 | Dict của list | `d.setdefault(r, []).append(w)` | chỉ thêm khi `r != -1` |

## 18. Đáp án tham khảo

**Bài 1:** `False` (vì `'a'` và `'c'` không thuộc hàng 1); phép hiệu là `{'a', 'c'}`.

In [25]:
print(set("Peace".lower()) <= set("qwertyuiop"))
print(set("Peace".lower()) - set("qwertyuiop"))

False
{'a', 'c'}


In [26]:
def answer_exercise_find_words(words):
    rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
    row_of = {}
    for i, row in enumerate(rows):
        for ch in row:
            row_of[ch] = i
    result = []
    for word in words:
        w = word.lower()
        target = row_of[w[0]]
        if all(row_of[ch] == target for ch in w):
            result.append(word)
    return result

def answer_count_per_row(words):
    counts = [0, 0, 0]
    for w in words:
        r = row_of_word(w)
        if r != -1:
            counts[r - 1] += 1
    return counts

def answer_longest_one_row_word(words):
    best = ""
    for w in words:
        if row_of_word(w) != -1 and len(w) > len(best):
            best = w
    return best

def answer_row_switches(word):
    w = word.lower()
    count = 0
    for i in range(1, len(w)):
        if ROW_OF[w[i]] != ROW_OF[w[i - 1]]:
            count += 1
    return count

def answer_group_by_row(words):
    groups = {}
    for w in words:
        r = row_of_word(w)
        if r != -1:
            groups.setdefault(r, []).append(w)
    return groups

assert answer_exercise_find_words(["Hello", "Alaska", "Dad", "Peace"]) == ["Alaska", "Dad"]
assert answer_count_per_row(["Hello", "Alaska", "Dad", "Peace", "type", "zxc"]) == [1, 2, 1]
assert answer_longest_one_row_word(["Hello", "Alaska", "Dad", "typewriter"]) == "typewriter"
assert answer_longest_one_row_word(["Hello"]) == ""
assert answer_row_switches("hello") == 3
assert answer_row_switches("Dad") == 0
assert answer_group_by_row(["Alaska", "Dad", "type", "Hello"]) == {2: ["Alaska", "Dad"], 1: ["type"]}
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
row_of = {ch: i for i, row in enumerate(rows) for ch in row}   # dựng 1 lần
result = []
for word in words:
    w = word.lower()                                # chuẩn hoá
    t = row_of[w[0]]                                # hàng mục tiêu
    if all(row_of[ch] == t for ch in w):            # mọi ký tự cùng hàng
        result.append(word)                         # trả bản GỐC
```

### Checklist
- [ ] Đã `lower()` trước khi tra cứu.
- [ ] Trả về từ **gốc**, đúng **thứ tự** ban đầu.
- [ ] Dùng `all` (mọi ký tự), không phải `any`.
- [ ] Bảng tra dựng một lần, ngoài vòng lặp.
- [ ] Time `O(N)` với `N` = tổng số ký tự; Space phụ `O(1)`.